<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building RAG Agents with LLMs</b></font></h1>
<h2><b>Notebook 4: </b>Running State Chain</h2>
<br>

이전 노트북에서는 runnable에 관한 핵심 LangChain Expression Language(LCEL) 내용을 소개했습니다. 이제 내부 추론과 외부 추론, 그리고 이를 가능하게 하는 파이프라인 개발에 익숙해졌을 것입니다! 이 노트북에서는 더 복잡한 대화 관리 전략을 오케스트레이션하고 장문 문서 추론을 시작할 수 있게 해 주는 고급 패러다임으로 나아갑니다.
<br>

### **학습 목표:**

- runnable을 활용해 흥미로운 LLM 시스템을 오케스트레이션하는 방법을 배웁니다.  
- running state chain이 대화 관리와 반복적 의사 결정에 어떻게 사용될 수 있는지 이해합니다.

<br>

### **생각해 볼 질문:**

- 환경에서 계속 입력을 받지 않는 단일 모듈 형태의 running state chain이 쓸모 있는 경우가 있을까요?
- JSON 예측이 사실 꽤 잘 동작한다는 것을 눈치채셨을 겁니다. 하지만 질문과 JSON 형식의 복잡도에 따라 항상 잘 동작하지는 않을 수 있습니다. 이와 관련해 어떤 문제가 생길 것으로 예상하나요?
- running state chain의 일부로 프롬프트를 완전히 교체하는 데 어떤 접근법을 생각해 볼 수 있을까요?

<br>

### **환경 설정:**

In [ ]:
## Necessary for Colab, not necessary for course environment
# %pip install -q langchain langchain-nvidia-ai-endpoints gradio

# import os
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."

from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
pprint = partial(console.print, style=base_style)

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
# ChatNVIDIA.get_available_models()

In [ ]:
## Useful utility method for printing intermediate states
from langchain_core.runnables import RunnableLambda
from functools import partial

def RPrint(preface="State: "):
    def print_and_return(x, preface=""):
        print(f"{preface}{x}")
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

def PPrint(preface="State: "):
    def print_and_return(x, preface=""):
        pprint(preface, x)
        return x
    return RunnableLambda(partial(print_and_return, preface=preface))

----

<br>

## **Part 1:** 변수가 계속 흐르게 하기

이전 예제에서는 상태를 **생성**하고, **변형**하고, **소비**함으로써 독립적인 체인에 흥미로운 로직을 구현할 수 있었습니다. 이 상태들은 설명적인 키와 유용한 값을 가진 딕셔너리로 전달되었고, 그 값들은 후속 루틴이 동작하는 데 필요한 정보를 제공하는 데 사용되었습니다!

**지난 노트북의 zero-shot 분류 예제를 떠올려 보세요:**

In [ ]:
# %%time
## ^^ This notebook is timed, which will print out how long it all took

from langchain_core.runnables import RunnableLambda
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from typing import List, Union
from operator import itemgetter

## Zero-shot classification prompt and chain
sys_msg = (
    "Choose the most likely topic classification given the sentence as context."
    " Only one word, no explanation.\n[Options : {options}]"
)

zsc_prompt = ChatPromptTemplate.from_messages([
    ("system", sys_msg),
    ("user", "[[{input}]]"),
])

## Define your simple instruct_model
instruct_chat = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})
instruct_llm = instruct_chat | StrOutputParser()

zsc_chain = zsc_prompt | instruct_llm

## Function that just prints out the first word of the output. With early stopping bind
def zsc_call(input, options=["car", "boat", "airplane", "bike"]):
    return zsc_chain.invoke({"input" : input, "options" : options}).split()[0]

print("-" * 80)
print(zsc_call("Should I take the next exit, or keep going to the next one?"))

print("-" * 80)
print(zsc_call("I get seasick, so I think I'll pass on the trip"))

print("-" * 80)
print(zsc_call("I'm scared of heights, so flying probably isn't for me"))

<br>

이 체인은 매우 사용하기 쉽게 만드는 몇 가지 설계 결정을 담고 있으며, 그중 핵심은 다음과 같습니다:

**함수처럼 동작하기를 원하므로, 출력을 생성하고 반환하는 것만 하면 됩니다.**

이 덕분에 체인을 더 큰 체인 시스템의 모듈로 포함하기가 매우 자연스러워집니다. 예를 들어 다음 체인은 문자열을 받아 가장 가능성 높은 주제를 추출한 다음, 그 주제를 바탕으로 새로운 문장을 생성합니다:

In [ ]:
%%time
## ^^ This notebook is timed, which will print out how long it all took
gen_prompt = ChatPromptTemplate.from_template(
    "Make a new sentence about the the following topic: {topic}. Be creative!"
)

gen_chain = gen_prompt | instruct_llm

input_msg = "I get seasick, so I think I'll pass on the trip"
options = ["car", "boat", "airplane", "bike"]

chain = (
    ## -> {"input", "options"}
    {'topic' : zsc_chain}
    | PPrint()
    ## -> {**, "topic"}
    | gen_chain
    ## -> string
)

chain.invoke({"input" : input_msg, "options" : options})

<br>

하지만 정보를 계속 흐르게 하고 싶을 때는 다소 문제가 됩니다. 응답을 생성하는 과정에서 topic과 input 변수를 잃어버리기 때문입니다. 출력과 입력 모두를 가지고 무언가를 하고 싶다면 두 변수가 모두 통과하도록 보장할 방법이 필요합니다.

다행히 매핑 runnable(즉, 딕셔너리로부터 해석되거나 수동 `RunnableMap`을 사용한)을 사용하면, 체인의 출력을 하나의 키에만 할당하고 나머지 키는 원하는 대로 전파되게 하여 두 변수를 모두 통과시킬 수 있습니다. 또는 `RunnableAssign`을 사용해 상태를 소비하는 체인의 출력을 기본적으로 입력 딕셔너리와 병합할 수도 있습니다.

이 기법을 사용하면 체인 시스템을 통해 원하는 것은 무엇이든 전파할 수 있습니다:

In [ ]:
%%time
## ^^ This notebook is timed, which will print out how long it all took

from langchain_core.runnables import RunnableBranch, RunnablePassthrough
from langchain_core.runnables.passthrough import RunnableAssign
from functools import partial

big_chain = (
    PPrint()
    ## Manual mapping. Can be useful sometimes and inside branch chains
    | {'input' : lambda d: d.get('input'), 'topic' : zsc_chain}
    | PPrint()
    ## RunnableAssign passing. Better for running state chains by default
    | RunnableAssign({'generation' : gen_chain})
    | PPrint()
    ## Using the input and generation together
    | RunnableAssign({'combination' : (
        ChatPromptTemplate.from_template(
            "Consider the following passages:"
            "\nP1: {input}"
            "\nP2: {generation}"
            "\n\nCombine the ideas from both sentences into one simple one."
        )
        | instruct_llm
    )})
)

output = big_chain.invoke({
    "input" : "I get seasick, so I think I'll pass on the trip",
    "options" : ["car", "boat", "airplane", "bike", "unknown"]
})
pprint("Final Output: ", output)

----

<br>

## **Part 2:** Running State Chain

위 예제는 그저 장난감 예제일 뿐이며, 오히려 내부 추론을 위해 많은 LLM 호출을 연결하는 것의 단점을 보여 줍니다. 하지만 체인을 통해 정보가 계속 흐르게 하는 능력은 유용한 상태 정보를 축적하거나 다중 패스로 동작하는 복잡한 체인을 만드는 데 매우 귀중합니다.

특히 매우 단순하지만 효과적인 체인이 **Running State Chain**으로, 다음 속성을 강제합니다:
- **"running state"** 는 시스템이 관심을 갖는 모든 변수를 담은 딕셔너리입니다.
- **"branch"** 는 running state를 끌어와 응답으로 축약할 수 있는 체인입니다.
- **branch**는 **RunnableAssign** 범위 안에서만 실행될 수 있으며, branch의 입력은 **running state**에서 와야 합니다.

> <img src="https://dli-lms.s3.amazonaws.com/assets/s-fx-15-v1/imgs/running_state_chain.png" width=1000px/>
<!-- > <img src="https://drive.google.com/uc?export=view&id=1Oo7AauYGj4dxepNReRG2JezmvQLyqXsN" width=1000px/> -->

running state chain 추상화는 상태 변수(속성)와 함수(메서드)를 가진 Python 클래스의 함수형 변형이라고 생각할 수 있습니다.
- 체인은 모든 기능을 감싸는 추상 클래스와 같습니다.
- running state는 (항상 접근 가능해야 하는) 속성과 같습니다.
- branch는 (어떤 속성을 사용할지 골라 쓸 수 있는) 클래스 메서드와 같습니다.
- `.invoke` 등의 과정은 branch들을 순서대로 실행하는 `__call__` 메서드와 같습니다.

**체인에 이 패러다임을 강제하면:**
- 상태 변수가 체인을 통해 계속 전파되어, 내부 로직이 필요한 것에 접근하고 나중에 사용할 상태 값을 축적할 수 있습니다.
- 체인의 출력을 다시 입력으로 넘겨서, running state를 계속 갱신하고 쌓아 가는 "while 루프" 스타일의 체인을 만들 수도 있습니다.

이 노트북의 나머지 부분에는 running state chain 추상화를 두 가지 추가 활용 사례, 즉 **지식 베이스(Knowledge Base)** 와 **데이터베이스 조회 챗봇**으로 구체화하는 두 개의 실습이 포함되어 있습니다.

----

<br>

## **Part 3:** Running State Chain으로 지식 베이스 구현하기

Running State Chain의 기본 구조와 원리를 이해했으니, 이 접근법을 더 복잡한 작업, 특히 상호작용을 통해 진화하는 동적 시스템을 만드는 데 어떻게 확장할 수 있는지 살펴볼 수 있습니다. 이 섹션에서는 **JSON 기반 슬롯 채우기(slot filling)** 로 축적되는 **지식 베이스**를 구현하는 데 집중합니다:

- **지식 베이스:** LLM이 추적해야 할 관련 정보의 저장소.
- **JSON 기반 슬롯 채우기:** 지시 튜닝된 모델에게 여러 슬롯을 가진 JSON 스타일 형식(딕셔너리를 포함할 수 있음)을 출력하도록 요청하고, LLM이 이 슬롯을 유용하고 관련 있는 정보로 채우도록 하는 기법.

<br>

#### **지식 베이스 정의하기**

반응성 있고 지능적인 시스템을 만들려면 입력을 처리할 뿐만 아니라 대화의 흐름 속에서 핵심 정보를 유지하고 갱신하는 방법이 필요합니다. 여기서 LangChain과 Pydantic의 조합이 중요해집니다. 인기 있는 Python 검증 라이브러리인 [**Pydantic**](https://docs.pydantic.dev/latest/)은 데이터 모델을 구조화하고 검증하는 데 핵심적인 역할을 합니다. Pydantic의 기능 중 하나로, 단순화된 문법과 깊이 있는 커스터마이징 옵션으로 객체(데이터, 클래스, 자기 자신 등)를 검증하는 구조화된 "model" 클래스를 제공합니다. 이 프레임워크는 LangChain 전반에서 사용되며, 데이터 강제 변환이 필요한 활용 사례에서 필수 구성 요소로 등장합니다.

"model"이 매우 잘하는 일 중 하나는 기대되는 인자와 이를 검증하는 특별한 방법을 갖춘 클래스를 정의하는 것입니다! 이 코스에서는 검증 스크립트에 크게 집중하지 않겠지만, 관심 있는 분은 [**Pydantic Validator 가이드**](https://docs.pydantic.dev/1.10/usage/validators/)부터 살펴보시면 됩니다(주제가 꽤 빠르게 깊어지긴 합니다). 우리 목적에는 `BaseModel` 클래스를 만들고 몇 개의 `Field` 변수를 정의하여 다음과 같이 구조화된 **지식 베이스**를 구성하면 됩니다:

In [ ]:
from pydantic import BaseModel, Field
from typing import Dict, Union, Optional

instruct_chat = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}})

class KnowledgeBase(BaseModel):
    ## Fields of the BaseModel, which will be validated/assigned when the knowledge base is constructed
    topic: str = Field('general', description="Current conversation topic")
    user_preferences: Dict[str, Union[str, int]] = Field({}, description="User preferences and choices")
    session_notes: list = Field([], description="Notes on the ongoing session")
    unresolved_queries: list = Field([], description="Unresolved user queries")
    action_items: list = Field([], description="Actionable items identified during the conversation")

print(repr(KnowledgeBase(topic = "Travel")))

<br>

이 접근법의 진정한 강점은 LangChain이 제공하는 추가적인 LLM 중심 기능을 활용 사례에 통합할 수 있다는 점에 있습니다. 그런 기능 중 하나가 `PydanticOutputParser`로, 자동 형식 지시문 생성 같은 기능으로 Pydantic 객체를 강화합니다.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

instruct_string = PydanticOutputParser(pydantic_object=KnowledgeBase).get_format_instructions()
pprint(instruct_string)

이 기능은 지식 베이스에 유효한 입력을 만들기 위한 지시문을 생성하며, 원하는 출력 형식의 구체적인 one-shot 예시를 제공하여 LLM을 돕습니다.

<br>

#### **Runnable 추출 모듈**

좋은 LLM 지시문을 생성하는 데 사용할 수 있는 이 Pydantic 객체가 있으니, Pydantic 클래스의 기능을 감싸고 지식 베이스의 프롬프팅, 생성, 갱신을 간소화하는 Runnable을 만들 수 있습니다:

In [ ]:
################################################################################
## Definition of RExtract
def RExtract(pydantic_class, llm, prompt):
    '''
    Runnable Extraction module
    Returns a knowledge dictionary populated by slot-filling extraction
    '''
    parser = PydanticOutputParser(pydantic_object=pydantic_class)
    instruct_merge = RunnableAssign({'format_instructions' : lambda x: parser.get_format_instructions()})
    def preparse(string):
        if '{' not in string: string = '{' + string
        if '}' not in string: string = string + '}'
        string = (string
            .replace("\\_", "_")
            .replace("\n", " ")
            .replace("\]", "]")
            .replace("\[", "[")
        )
        # print(string)  ## Good for diagnostics
        return string
    return instruct_merge | prompt | llm | preparse | parser

################################################################################
## Practical Use of RExtract

parser_prompt = ChatPromptTemplate.from_template(
    "Update the knowledge base: {format_instructions}. Only use information from the input."
    "\n\nNEW MESSAGE: {input}"
)

extractor = RExtract(KnowledgeBase, instruct_llm, parser_prompt)

knowledge = extractor.invoke({'input' : "I love flowers so much! The orchids are amazing! Can you buy me some?"})
pprint(knowledge)

<br>

LLM 예측의 모호한 특성 때문에, 특히 지시 따르기에 최적화되지 않은 모델에서는 이 과정이 실패할 수 있다는 점을 기억하세요! 이 과정에서는 지시를 잘 따르는 강력한 LLM과 함께 추가 검사 및 우아한 실패 처리 루틴을 갖추는 것이 중요합니다. 

<br>

#### **동적 지식 베이스 갱신**

마지막으로, 대화 내내 지식 베이스를 지속적으로 갱신하는 시스템을 만들 수 있습니다. 이는 지식 베이스의 현재 상태를 새로운 사용자 입력과 함께 시스템에 다시 넣어 계속 갱신하는 방식으로 이루어집니다.

다음은 이 방식이 세부 사항을 채우는 힘과, 채우기 성능이 일반 응답 성능만큼 좋을 것이라고 가정하는 것의 한계를 모두 보여 주는 예시 시스템입니다:

In [ ]:
class KnowledgeBase(BaseModel):
    firstname: str = Field('unknown', description="Chatting user's first name, unknown if unknown")
    lastname: str = Field('unknown', description="Chatting user's last name, unknown if unknown")
    location: str = Field('unknown', description="Where the user is located")
    summary: str = Field('unknown', description="Running summary of conversation. Update this with new input")
    response: str = Field('unknown', description="An ideal response to the user based on their new message")


parser_prompt = ChatPromptTemplate.from_template(
    "You are chatting with a user. The user just responded ('input'). Please update the knowledge base."
    " Record your response in the 'response' tag to continue the conversation."
    " Do not hallucinate any details, and make sure the knowledge base is not redundant."
    " Update the entries frequently to adapt to the conversation flow."
    "\n{format_instructions}"
    "\n\nOLD KNOWLEDGE BASE: {know_base}"
    "\n\nNEW MESSAGE: {input}"
    "\n\nNEW KNOWLEDGE BASE:"
)

## Use the course text model for extraction
instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}}) | StrOutputParser()

extractor = RExtract(KnowledgeBase, instruct_llm, parser_prompt)
info_update = RunnableAssign({'know_base' : extractor})

## Initialize the knowledge base and see what you get
state = {'know_base' : KnowledgeBase()}
state['input'] = "My name is Carmen Sandiego! Guess where I am! Hint: It's somewhere in the United States."
state = info_update.invoke(state)
pprint(state)

In [ ]:
state['input'] = "I'm in a place considered the birthplace of Jazz."
state = info_update.invoke(state)
pprint(state)

In [ ]:
state['input'] = "Yeah, I'm in New Orleans... How did you know?"
state = info_update.invoke(state)
pprint(state)

<br>

이 예제는 running state chain이 변화하는 맥락과 요구 사항을 가진 대화를 관리하는 데 효과적으로 활용될 수 있음을 보여 주며, 정교한 대화형 시스템을 개발하는 강력한 도구가 됩니다.

이 노트북의 다음 섹션에서는 **문서 지식 베이스**와 **데이터베이스 조회 챗봇**이라는 두 가지 구체적인 응용을 탐구하며 이 개념을 확장합니다.

----

<br>

## **Part 4: [실습]** 항공사 고객 서비스 봇

이 실습에서는 지금까지 배운 도구를 확장하여 단순하지만 효과적인 대화 관리 챗봇을 구현합니다. 이번 실습에서는 고객이 자신의 항공편에 대해 알아볼 수 있도록 돕는 항공사 지원 봇을 만들어 봅니다!

딕셔너리에서 고객 정보를 가져오는 간단한 데이터베이스 같은 인터페이스를 만들어 봅시다!

In [ ]:
#######################################################################################
## Function that can be queried for information. Implementation details not important
def get_flight_info(d: dict) -> str:
    """
    Example of a retrieval function which takes a dictionary as key. Resembles SQL DB Query
    """
    req_keys = ['first_name', 'last_name', 'confirmation']
    assert all((key in d) for key in req_keys), f"Expected dictionary with keys {req_keys}, got {d}"

    ## Static dataset. get_key and get_val can be used to work with it, and db is your variable
    keys = req_keys + ["departure", "destination", "departure_time", "arrival_time", "flight_day"]
    values = [
        ["Jane", "Doe", 12345, "San Jose", "New Orleans", "12:30 PM", "9:30 PM", "tomorrow"],
        ["John", "Smith", 54321, "New York", "Los Angeles", "8:00 AM", "11:00 AM", "Sunday"],
        ["Alice", "Johnson", 98765, "Chicago", "Miami", "7:00 PM", "11:00 PM", "next week"],
        ["Bob", "Brown", 56789, "Dallas", "Seattle", "1:00 PM", "4:00 PM", "yesterday"],
    ]
    get_key = lambda d: "|".join([d['first_name'], d['last_name'], str(d['confirmation'])])
    get_val = lambda l: {k:v for k,v in zip(keys, l)}
    db = {get_key(get_val(entry)) : get_val(entry) for entry in values}

    # Search for the matching entry
    data = db.get(get_key(d))
    if not data:
        return (
            f"Based on {req_keys} = {get_key(d)}) from your knowledge base, no info on the user flight was found."
            " This process happens every time new info is learned. If it's important, ask them to confirm this info."
        )
    return (
        f"{data['first_name']} {data['last_name']}'s flight from {data['departure']} to {data['destination']}"
        f" departs at {data['departure_time']} {data['flight_day']} and lands at {data['arrival_time']}."
    )

#######################################################################################
## Usage example. Actually important

print(get_flight_info({"first_name" : "Jane", "last_name" : "Doe", "confirmation" : 12345}))

In [ ]:
print(get_flight_info({"first_name" : "Alice", "last_name" : "Johnson", "confirmation" : 98765}))

In [ ]:
print(get_flight_info({"first_name" : "Bob", "last_name" : "Brown", "confirmation" : 27494}))

<br>

이 인터페이스는 두 가지 목적을 합리적으로 수행할 수 있기 때문에 살펴볼 만합니다:
- 사용자의 상황에 관한 외부 환경(데이터베이스)의 최신 정보를 제공하는 데 사용할 수 있습니다.
- 민감한 정보의 무단 공개를 막는 강력한 게이팅 메커니즘으로도 사용할 수 있습니다(무단 공개는 매우 나쁜 일이니까요).

우리 네트워크가 이런 인터페이스에 접근할 수 있다면, 사용자를 대신해 이 정보를 조회하고 가져올 수 있을 것입니다! 예를 들면:

In [ ]:
external_prompt = ChatPromptTemplate.from_template(
    "You are a SkyFlow chatbot, and you are helping a customer with their issue."
    " Please help them with their question, remembering that your job is to represent SkyFlow airlines."
    " Assume SkyFlow uses industry-average practices regarding arrival times, operations, etc."
    " (This is a trade secret. Do not disclose)."  ## soft reinforcement
    " Please keep your discussion short and sweet if possible. Avoid saying hello unless necessary."
    " The following is some context that may be useful in answering the question."
    "\n\nContext: {context}"
    "\n\nUser: {input}"
)

basic_chain = external_prompt | instruct_llm

basic_chain.invoke({
    'input' : 'Can you please tell me when I need to get to the airport?',
    'context' : get_flight_info({"first_name" : "Jane", "last_name" : "Doe", "confirmation" : 12345}),
})

<br>

이것만으로도 충분히 흥미롭지만, 실제 환경에서 이 시스템이 동작하게 하려면 어떻게 해야 할까요? 위의 KnowledgeBase 방식을 사용해 이런 정보를 다음과 같이 제공할 수 있습니다:

In [ ]:
from pydantic import BaseModel, Field
from typing import Dict, Union

class KnowledgeBase(BaseModel):
    first_name: str = Field('unknown', description="Chatting user's first name, `unknown` if unknown")
    last_name: str = Field('unknown', description="Chatting user's last name, `unknown` if unknown")
    confirmation: int = Field(-1, description="Flight Confirmation Number, `-1` if unknown")
    discussion_summary: str = Field("", description="Summary of discussion so far, including locations, issues, etc.")
    open_problems: list = Field([], description="Topics that have not been resolved yet")
    current_goals: list = Field([], description="Current goal for the agent to address")

def get_key_fn(base: BaseModel) -> dict:
    '''Given a dictionary with a knowledge base, return a key for get_flight_info'''
    return {  ## More automatic options possible, but this is more explicit
        'first_name' : base.first_name,
        'last_name' : base.last_name,
        'confirmation' : base.confirmation,
    }

know_base = KnowledgeBase(first_name = "Jane", last_name = "Doe", confirmation = 12345)

# get_flight_info(get_key_fn(know_base))

get_key = RunnableLambda(get_key_fn)
(get_key | get_flight_info).invoke(know_base)

<br>

### **목표:**

사용자가 대화 교환의 일부로 다음 함수 호출을 자연스럽게 유발할 수 있게 하고 싶습니다:

```python
get_flight_info({"first_name" : "Jane", "last_name" : "Doe", "confirmation" : 12345}) ->
    "Jane Doe's flight from San Jose to New Orleans departs at 12:30 PM tomorrow and lands at 9:30 PM."
```

다음과 같은 지식 베이스 문법을 사용할 수 있도록 `RExtract`가 제공됩니다:
```python
known_info = KnowledgeBase()
extractor = RExtract(KnowledgeBase, InstructLLM(), parser_prompt)
results = extractor.invoke({'info_base' : known_info, 'input' : 'My message'})
known_info = results['info_base']
```

**다음 기능을 구현하는 챗봇을 설계하세요:**
- 봇은 가벼운 대화로 시작해야 하며, 개인 정보 접근이 필요 없는 민감하지 않은 질문은 도와줄 수 있습니다.
- 사용자가 (실질적으로나 법적으로) 데이터베이스로 보호된 것들에 대해 묻기 시작하면, 관련 정보를 제공해야 한다고 사용자에게 알려야 합니다.
- 조회에 성공하면 에이전트는 데이터베이스로 보호된 정보에 대해 이야기할 수 있게 됩니다.

**이는 다음을 포함한 다양한 기법으로 구현할 수 있습니다:**
- **프롬프트 엔지니어링과 컨텍스트 파싱**: 전체 채팅 프롬프트는 대체로 그대로 두되, 컨텍스트를 조작하여 에이전트의 동작을 바꿉니다. 예를 들어 DB 조회 실패를 *`"Information could not be retrieved with keys {...}. Please ask the user for clarification or help them with known information."`* 처럼 문제 해결 방법에 대한 자연어 지시문 주입으로 바꿀 수 있습니다.
- **"프롬프트 전달(Prompt Passing)"**: 활성 프롬프트를 상태 변수로 전달하고, 모니터링 체인이 이를 덮어쓸 수 있게 합니다.
- [**`RunnableBranch`**](https://reference.langchain.com/python/langchain-core/runnables/branch/RunnableBranch) 같은 **분기 체인** 또는 조건부 라우팅 메커니즘을 구현하는 더 커스텀한 해결책.
    - [`RunnableBranch`](https://reference.langchain.com/python/langchain-core/runnables/branch/RunnableBranch)의 경우 다음과 같은 스타일의 `switch` 문법을 사용합니다:
        ```python
        from langchain_core.runnables import RunnableBranch
        RunnableBranch(
            ((lambda x: 1 in x), RPrint("Has 1 (didn't check 2): ")),
            ((lambda x: 2 in x), RPrint("Has 2 (not 1 though): ")),
            RPrint("Has neither 1 not 2: ")
        ).invoke([2, 1, 3]);  ## -> Has 1 (didn't check 2): [2, 1, 3]
        ```

작업에 도움이 될 만한 몇 가지 프롬프트와 Gradio 루프가 제공되지만, 현재 에이전트는 그저 환각(hallucination)을 일으킬 것입니다! 관련 정보를 조회하도록 내부 체인을 구현하세요. 구현하기 전에 모델의 기본 동작을 살펴보고, 어떻게 환각을 일으키거나 정보를 잊어버리는지 확인해 보세요.

In [ ]:
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableMap,       ## Wrap an implicit "dictionary" runnable
    RunnablePassthrough,
)
from langchain_core.runnables.passthrough import RunnableAssign

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import BaseMessage, SystemMessage, ChatMessage, AIMessage
from typing import Iterable
import gradio as gr

external_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a chatbot for SkyFlow Airlines, and you are helping a customer with their issue."
        " Please chat with them! Stay concise and clear!"
        " Your running knowledge base is: {know_base}."
        " This is for you only; Do not mention it!"
        " \nUsing that, we retrieved the following: {context}\n"
        " If they provide info and the retrieval fails, ask to confirm their first/last name and confirmation."
        " Do not ask them any other personal info."
        " If it's not important to know about their flight, do not ask."
        " The checking happens automatically; you cannot check manually."
    )),
    ("assistant", "{output}"),
    ("user", "{input}"),
])

##########################################################################
## Knowledge Base Things

class KnowledgeBase(BaseModel):
    first_name: str = Field('unknown', description="Chatting user's first name, `unknown` if unknown")
    last_name: str = Field('unknown', description="Chatting user's last name, `unknown` if unknown")
    confirmation: Optional[int] = Field(None, description="Flight Confirmation Number, `-1` if unknown")
    discussion_summary: str = Field("", description="Summary of discussion so far, including locations, issues, etc.")
    open_problems: str = Field("", description="Topics that have not been resolved yet")
    current_goals: str = Field("", description="Current goal for the agent to address")

parser_prompt = ChatPromptTemplate.from_template(
    "You are a chat assistant representing the airline SkyFlow, and are trying to track info about the conversation."
    " You have just received a message from the user. Please fill in the schema based on the chat."
    "\n\n{format_instructions}"
    "\n\nOLD KNOWLEDGE BASE: {know_base}"
    "\n\nASSISTANT RESPONSE: {output}"
    "\n\nUSER MESSAGE: {input}"
    "\n\nNEW KNOWLEDGE BASE: "
)

## Your goal is to invoke the following through natural conversation
# get_flight_info({"first_name" : "Jane", "last_name" : "Doe", "confirmation" : 12345}) ->
#     "Jane Doe's flight from San Jose to New Orleans departs at 12:30 PM tomorrow and lands at 9:30 PM."

chat_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}}) | StrOutputParser()
instruct_llm = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", timeout=300, model_kwargs={"chat_template_kwargs": {"enable_thinking": False}}) | StrOutputParser()

external_chain = external_prompt | chat_llm

#####################################################################################
## START TODO: Define the extractor and internal chain to satisfy the objective

## TODO: Make a chain that will populate your knowledge base based on provided context
knowbase_getter = lambda x: KnowledgeBase()

## TODO: Make a chain to pull d["know_base"] and outputs a retrieval from db
database_getter = lambda x: "Not implemented"

## These components integrate to make your internal chain
internal_chain = (
    RunnableAssign({'know_base' : knowbase_getter})
    | RunnableAssign({'context' : database_getter})
)

## END TODO
#####################################################################################

state = {'know_base' : KnowledgeBase()}

def chat_gen(message, history=[], return_buffer=True):

    ## Pulling in, updating, and printing the state
    global state
    state['input'] = message
    state['history'] = history

    ## Logic to extract the last assistant message from dictionary history
    last_assistant_msg = ""
    for entry in reversed(history):
        if entry.get("role") == "assistant":
            last_assistant_msg = entry.get("content", "")
            break
    state['output'] = last_assistant_msg

    ## Generating the new state from the internal chain
    state = internal_chain.invoke(state)
    
    print("\n--- Internal State Update ---")
    pprint({k:v for k,v in state.items() if k != "history"})
    
    ## Streaming the results
    buffer = ""
    for token in external_chain.stream(state):
        buffer += token
        yield buffer if return_buffer else token

def queue_fake_streaming_gradio(chat_stream, history = [], max_questions=8):

    ## Mimic of the gradio initialization routine, where a set of starter messages can be printed off
    for entry in history:
        role = entry.get("role", "unknown").capitalize()
        print(f"\n[ {role} ]:", entry.get("content"))

    ## Mimic of the gradio loop with an initial message from the agent.
    for _ in range(max_questions):
        message = input("\n[ Human ]: ")
        # Append User Message
        history.append({"role": "user", "content": message})
        
        print("\n[ Agent ]: ")
        full_response = ""
        for token in chat_stream(message, history, return_buffer=False):
            print(token, end='')
            full_response += token
        
        # Append Assistant Message
        history.append({"role": "assistant", "content": full_response})
        print("\n")

## history is of format [[User response 0, Bot response 0], ...]
chat_history = [{"role": "assistant", "content": "Hello! I'm your SkyFlow agent! How can I help you?"}]

queue_fake_streaming_gradio(
    chat_stream = chat_gen,
    history = chat_history
)

In [ ]:
# state = {'know_base' : KnowledgeBase()}

# chatbot = gr.Chatbot(value=chat_history)
# demo = gr.ChatInterface(chat_gen, chatbot=chatbot).queue().launch(debug=True, share=True)

<br>

----

<br>

**참고:**
- 예외 발생 후 Gradio 인터페이스가 멈추면 STOP 버튼을 명시적으로 누르고 다시 실행해야 할 수 있습니다. 이는 알려진 Jupyter Notebook 환경 문제이며, 전용 Gradio 실행 파일에서는 발생하지 않습니다.
- **빠른 참고를 위해 채팅 지시문을 여기에 다시 적어 둡니다:**
```python
## Your goal is to invoke the following through natural conversation
get_flight_info({
    "first_name" : "Jane",
    "last_name" : "Doe",
    "confirmation" : 12345,
}) -> "Jane Doe's flight from San Jose to New Orleans departs at 12:30 PM tomorrow and lands at 9:30 PM."
```
- **시스템이 동작하는지 확인하려면 다음과 같은 대화를 시도해 볼 수 있습니다:**
```
> How's it going?
> Can you tell me a bit about skyflow?
> Can you tell me about my flight?
> My name is Jane Doe and my flight confirmation is 12345
> Can you tell me when I should get to my flight?
```
- **실습 정답은 Solutions 디렉터리에서 찾을 수 있습니다.** 정답이 제공되는 첫 번째 실습이며, 이후 노트북의 추가 실습 정답도 그곳에서 찾을 수 있습니다.

-----

<br>

## **Part 5:** 마무리

이 노트북의 목표는 지식 베이스와 running state chain의 활용을 중심으로 더 고급 LangChain 내용을 소개하는 것이었습니다! 이번 실습은 꽤 복잡했으니, 완료하신 것을 축하합니다!

### <font color="#76b900">**수고하셨습니다!**</font>

### **다음 단계:**
1. **[선택]** 노트북 상단의 **"생각해 볼 질문" 섹션**을 다시 읽고 가능한 답을 생각해 보세요.

---

<div style="width: 55%%; background-color: white; margin-top: 50px;"><center><a href="https://www.nvidia.com/en-us/training/"><img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png" width="300" /></a></center></div>